# Session 8 — PDF RAG with FAISS

This notebook builds a minimal retrieval-augmented generation pipeline over generic handbook and benefits PDFs.

## Learning Goals

- understand why PDFs need preprocessing
- inspect extracted document text
- chunk documents with overlap
- build a FAISS index over embeddings
- compare no-RAG vs RAG answers

In [7]:
import json
import os
import sys
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from sentence_transformers import SentenceTransformer

load_dotenv()


OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_ORG_ID = os.getenv('OPENAI_ORG_ID')
OPENAI_PROJECT_ID = os.getenv('OPENAI_PROJECT_ID')

In [50]:
sys.path.insert(0, str(Path().cwd().parent))
# Path(os.__file__).parent

data_dir = Path().cwd() / '../data'
pdf_dir = (data_dir / 'pdfs').resolve()
questions_path = data_dir / 'questions.json'


print(f"PDF Directory: {pdf_dir}")
print('pdf_count =', len(list(pdf_dir.glob('*.pdf'))))

PDF Directory: C:\git\mdsiprojects\anlpaut2026\material\Session 8\data\pdfs
pdf_count = 4


In [54]:
from helpers.pdf_utils import extract_pdf_text, join_pages
from helpers.rag_utils import chunk_text, package_chunks, build_faiss_index, search_index, build_grounded_prompt

## 1. Load the PDF Corpus

In [55]:
pdf_paths = sorted(pdf_dir.glob('*.pdf'))
for path in pdf_paths:
    print(path.name)

employee_handbook.pdf
Northwind_Health_Plus_Benefits_Details.pdf
Northwind_Standard_Benefits_Details.pdf
PerksPlus.pdf


## 2. Extract Text

PDFs are not directly useful to an LLM. We first need to extract text from each page.

In [56]:
documents = []
for pdf_path in pdf_paths:
    pages = extract_pdf_text(pdf_path)
    full_text = join_pages(pages)
    documents.append({
        'source_file': pdf_path.name,
        'pages': pages,
        'text': full_text,
    })

for doc in documents:
    print(doc['source_file'], 'pages =', len(doc['pages']), 'chars =', len(doc['text']))

employee_handbook.pdf pages = 11 chars = 16070
Northwind_Health_Plus_Benefits_Details.pdf pages = 109 chars = 257074
Northwind_Standard_Benefits_Details.pdf pages = 104 chars = 252053
PerksPlus.pdf pages = 4 chars = 2856


In [57]:
sample_doc = documents[0]
print(sample_doc['source_file'])
print(sample_doc['text'][:1200])

employee_handbook.pdf
Contoso Electronics 
Employee Handbook

This document contains information generated using a language model (Azure OpenAI). The 
information contained in this document is only for demonstration purposes and does not 
reflect the opinions or beliefs of Microsoft. Microsoft makes no representations or 
warranties of any kind, express or implied, about the completeness, accuracy, reliability, 
suitability or availability with respect to the information contained in this document.  
All rights reserved to Microsoft

Contoso Electronics Employee Handbook 
Last Updated: 2023-03-05 
 
Contoso Electronics is a leader in the aerospace industry, providing advanced electronic 
components for both commercial and military aircraft. We specialize in creating cutting-
edge systems that are both reliable and efficient. Our mission is to provide the highest 
quality aircraft components to our customers, while maintaining a commitment to safety 
and excellence. We are proud to have

## 3. Chunk the Documents

Chunking breaks long text into smaller pieces. Overlap helps preserve context across chunk boundaries.

In [59]:
all_chunks = []
for doc in documents:
    chunks = chunk_text(doc['text'], chunk_size=500, overlap=100)
    all_chunks.extend(package_chunks(chunks, doc['source_file']))

print('total chunks =', len(all_chunks))
all_chunks[:5]

total chunks = 1299


[{'chunk_id': 0,
  'source_file': 'employee_handbook.pdf',
  'text': 'Contoso Electronics Employee Handbook This document contains information generated using a language model (Azure OpenAI). The information contained in this document is only for demonstration purposes and does not reflect the opinions or beliefs of Microsoft. Microsoft makes no representations or warranties of any kind, express or implied, about the completeness, accuracy, reliability, suitability or availability with respect to the information contained in this document. All rights reserved to M'},
 {'chunk_id': 1,
  'source_file': 'employee_handbook.pdf',
  'text': 'or availability with respect to the information contained in this document. All rights reserved to Microsoft Contoso Electronics Employee Handbook Last Updated: 2023-03-05 Contoso Electronics is a leader in the aerospace industry, providing advanced electronic components for both commercial and military aircraft. We specialize in creating cutting- edge s

## 4. Create Embeddings

Session 8 is OpenAI-first, so the preferred path uses OpenAI embeddings. A local sentence-transformers fallback is included for offline work.

In [62]:
chunk_texts = [chunk['text'] for chunk in all_chunks]

if OPENAI_API_KEY:
    client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)
    embedding_response = client.embeddings.create(
        model='text-embedding-3-small',
        input=chunk_texts
    )
    chunk_embeddings = np.array([item.embedding for item in embedding_response.data], dtype='float32')
    embedding_source = 'OpenAI embeddings'
else:
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    chunk_embeddings = embedding_model.encode(chunk_texts, convert_to_numpy=True).astype('float32')
    embedding_source = 'sentence-transformers fallback'

print(embedding_source)
print(chunk_embeddings.shape)
print(chunk_embeddings[:2])

OpenAI embeddings
(1299, 1536)
[[0.00898743 0.02018738 0.0213623  ... 0.01358795 0.00297928 0.01649475]
 [0.02627563 0.02467346 0.04403687 ... 0.02215576 0.03384399 0.01976013]]


## 5. Build a FAISS Index

In [61]:
index = build_faiss_index(chunk_embeddings)
print(index.ntotal)

1299


## 6. Retrieve Relevant Chunks

In [68]:
question = 'Which health plan appears to offer broader benefits coverage?'

if OPENAI_API_KEY:
    query_embedding = np.array(
        OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID).embeddings.create(
            model='text-embedding-3-small',
            input=[question]
        ).data[0].embedding,
        dtype='float32'
    )
else:
    query_embedding = embedding_model.encode([question], convert_to_numpy=True).astype('float32')[0]

distances, indices = search_index(index, query_embedding, top_k=5)
retrieved_chunks = [all_chunks[idx] for idx in indices[0]]
retrieved_chunks

[{'chunk_id': 446,
  'source_file': 'Northwind_Health_Plus_Benefits_Details.pdf',
  'text': 'as any other insurance, health plan, or other coverage which provides benefits and services for medical care that is also provided under the Northwind Health Plus plan. This includes, but is not limited to, Medicare, TRICARE, Medicaid, employer-sponsored plans, and government-sponsored plans. When you have Other Coverage, Northwind Health Plus follows something called “Coordination of Benefits” (COB). This means that Northwind Health Plus coordinates its benefits with the Other Coverage in orde'},
 {'chunk_id': 523,
  'source_file': 'Northwind_Health_Plus_Benefits_Details.pdf',
  'text': 's. The plan must also provide detailed information to participants, such as a Summary Plan Description (SPD), which outlines plan provisions and benefits. Under the ACA, Northwind Health Plus must provide essential health benefits, such as ambulatory patient services, hospitalization, maternity and newborn car

In [69]:
for chunk in retrieved_chunks:
    print(f"[{chunk['source_file']} | chunk {chunk['chunk_id']}]")
    print(chunk['text'][:700])
    print('-' * 80)

[Northwind_Health_Plus_Benefits_Details.pdf | chunk 446]
as any other insurance, health plan, or other coverage which provides benefits and services for medical care that is also provided under the Northwind Health Plus plan. This includes, but is not limited to, Medicare, TRICARE, Medicaid, employer-sponsored plans, and government-sponsored plans. When you have Other Coverage, Northwind Health Plus follows something called “Coordination of Benefits” (COB). This means that Northwind Health Plus coordinates its benefits with the Other Coverage in orde
--------------------------------------------------------------------------------
[Northwind_Health_Plus_Benefits_Details.pdf | chunk 523]
s. The plan must also provide detailed information to participants, such as a Summary Plan Description (SPD), which outlines plan provisions and benefits. Under the ACA, Northwind Health Plus must provide essential health benefits, such as ambulatory patient services, hospitalization, maternity and newbo

## 7. Build a Grounded Prompt

In [71]:
grounded_prompt = build_grounded_prompt(question, retrieved_chunks)
print(grounded_prompt[:5000])

Answer the question using only the retrieved context below. If the answer is not in the context, say you do not have enough information.

Question: Which health plan appears to offer broader benefits coverage?

Retrieved context:
[Northwind_Health_Plus_Benefits_Details.pdf | chunk 446] as any other insurance, health plan, or other coverage which provides benefits and services for medical care that is also provided under the Northwind Health Plus plan. This includes, but is not limited to, Medicare, TRICARE, Medicaid, employer-sponsored plans, and government-sponsored plans. When you have Other Coverage, Northwind Health Plus follows something called “Coordination of Benefits” (COB). This means that Northwind Health Plus coordinates its benefits with the Other Coverage in orde

[Northwind_Health_Plus_Benefits_Details.pdf | chunk 523] s. The plan must also provide detailed information to participants, such as a Summary Plan Description (SPD), which outlines plan provisions and benefits. 

## 8. Compare No-RAG vs RAG

These cells are optional if you do not have live API access. The comparison is most useful with a real model.

In [73]:
with open(questions_path, 'r', encoding='utf-8') as f:
    questions = json.load(f)

questions[:6]

[{'id': 'q1',
  'question': 'What does the employee handbook say about annual leave or vacation leave?',
  'expected_signal': 'The answer should reference leave or vacation policy from the handbook.',
  'source_docs': ['employee_handbook.pdf']},
 {'id': 'q2',
  'question': 'Which health plan appears to offer broader benefits coverage, the standard plan or the health plus plan?',
  'expected_signal': 'The answer should compare the two Northwind plans using retrieved evidence rather than guessing.',
  'source_docs': ['Northwind_Standard_Benefits_Details.pdf',
   'Northwind_Health_Plus_Benefits_Details.pdf']},
 {'id': 'q3',
  'question': 'What kinds of perks are described in the PerksPlus document?',
  'expected_signal': 'The answer should summarize perks from the PerksPlus PDF.',
  'source_docs': ['PerksPlus.pdf']},
 {'id': 'q4',
  'question': 'Does the handbook mention remote work or working from home?',
  'expected_signal': 'The answer should rely on retrieved handbook text and avoid i

In [74]:
# Requires OPENAI_API_KEY for live generation.
# Skip this cell if you do not have live API access.

client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)

demo_question = questions[0]['question']

no_rag = client.responses.create(
    model='gpt-4o-mini',
    instructions='Answer concisely.',
    input=demo_question,
)

rag = client.responses.create(
    model='gpt-4o-mini',
    instructions='Use the provided context only. If the answer is not in the context, say so.',
    input=grounded_prompt,
)

display(Markdown('### No RAG'))
display(Markdown(no_rag.output_text))
display(Markdown('### RAG'))
display(Markdown(rag.output_text))

### No RAG

The employee handbook typically outlines details such as the accrual rate of leave, how to request it, any caps on leave balances, and specific eligibility criteria. For accurate information, please refer directly to your company's handbook.

### RAG

Based on the retrieved context, the Northwind Health Plus plan appears to offer broader benefits coverage compared to the Northwind Standard plan. The Northwind Health Plus plan must provide essential health benefits under the ACA, including a wide range of services like ambulatory patient services, hospitalization, and mental health services, as well as preventive services without cost sharing. The Northwind Standard plan also offers coverage but is described as comprehensive without the same emphasis on essential health benefits. Therefore, Northwind Health Plus seems to have a broader coverage.

## 9. Extensions

- **OpenAI Vector Store:** managed alternative when you want OpenAI to handle chunking, embedding, and indexing.
- **Hybrid search:** production systems often combine keyword and dense retrieval.
- **Agentic RAG:** in Session 9, retrieval can become a tool inside an agent loop rather than a one-shot pipeline.